
# Dublin Bikes: Clustering and Time Series Forecasting

## CA1 — Machine Learning for Business

This project analyses historical Dublin Bikes data to explore bike availability patterns across stations and investigate changes in bike availability over time

The project will compare two clustering algorithms and evaluate a time series forecasting model


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
data_folder = Path("../data")

In [3]:
print(data_folder.exists())

True



## 2. Data Loading

The six monthly CSV files are loaded using Pandas I will check the number of records in each file before combining the data


In [4]:

csv_files = sorted(data_folder.glob("*.csv"))

dataframes = [pd.read_csv(file) for file in csv_files]

for file, df in zip(csv_files, dataframes):
    print(file.name, "-", len(df), "rows")


dublin-bikes_AUGUST.csv - 639029 rows
dublin-bikes_JULY.csv - 601170 rows
dublin-bikes_JUNE.csv - 606948 rows
dublin-bikes_MAY.csv - 640308 rows
dublin-bikes_OCTOBER.csv - 642050 rows
dublin-bikes_SEPTEMBER.csv - 622950 rows



## 3. Data Structure

In this step, we check the column names and number of columns in each monthly file, this helps confirm whether the files have the same structure before combining them 

In [5]:

for file, df in zip(csv_files, dataframes):
    print("\n", file.name)
    print("Number of columns:", len(df.columns))
    print(list(df.columns))



 dublin-bikes_AUGUST.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_JULY.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_JUNE.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'region_id', 'capacity']

 dublin-bikes_MAY.csv
Number of columns: 15
['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'short_name', 'address', 'lat', 'lon', 'reg


## 4. Combining the Monthly Data

The six monthly files have the same 15 columns I can combine them into one dataset to analyse the period from May to October 2025


In [6]:

bikes = pd.concat(dataframes, ignore_index=True)

print("Dataset shape:", bikes.shape)
print("Number of stations:", bikes["station_id"].nunique())
print("Date range:", bikes["last_reported"].min(), "to", bikes["last_reported"].max())


Dataset shape: (3752455, 15)
Number of stations: 115
Date range: 2025-05-01 00:05:00 to 2025-10-31 23:55:00



## 5. Data Quality Check

Before analysing the data, I check for missing values and duplicate rows, this helps identify data quality issues that may affect the results


In [7]:

print("Missing values per column:")
print(bikes.isnull().sum())

print("\nNumber of duplicate rows:", bikes.duplicated().sum())


Missing values per column:
system_id                    0
last_reported                0
station_id                   0
num_bikes_available          0
num_docks_available          0
is_installed                 0
is_renting                   0
is_returning                 0
name                         0
short_name             3752455
address                      0
lat                          0
lon                          0
region_id              3752455
capacity                     0
dtype: int64

Number of duplicate rows: 0



## 6. Data Consistency Check
I check whether the available bikes and available docks add up to the station capacity .. Differences may indicate inconsistent records that need further investigation


In [8]:

bikes["availability_total"] = (
    bikes["num_bikes_available"] + bikes["num_docks_available"]
)

print("Records where availability does not match capacity:")
print((bikes["availability_total"] != bikes["capacity"]).sum())


Records where availability does not match capacity:
196399


In [9]:

difference = bikes["availability_total"] - bikes["capacity"]

print("Difference summary:")
print(difference.describe())

print("\nMost common differences:")
print(difference.value_counts().head(10))


Difference summary:
count    3.752455e+06
mean    -3.774883e-01
std      3.515206e+00
min     -4.000000e+01
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.000000e+00
dtype: float64

Most common differences:
 0     3556056
-1      132487
-40      27715
-2       19685
-5        6712
-3        4954
-20       1631
-4        1203
-30       1120
-6         301
Name: count, dtype: int64


In [10]:

print(
    bikes.loc[
        difference == -40,
        ["last_reported", "station_id", "num_bikes_available",
         "num_docks_available", "capacity"]
    ].head(10)
)


            last_reported  station_id  num_bikes_available  \
29    2025-08-01 00:05:00          51                    0   
147   2025-08-01 00:15:00          51                    0   
266   2025-08-01 00:25:00          51                    0   
379   2025-08-01 00:35:00          51                    0   
497   2025-08-01 00:45:00          51                    0   
614   2025-08-01 00:55:00          51                    0   
727   2025-08-01 01:05:00          51                    0   
843   2025-08-01 01:15:00          51                    0   
956   2025-08-01 01:25:00          51                    0   
1072  2025-08-01 01:35:00          51                    0   

      num_docks_available  capacity  
29                      0        40  
147                     0        40  
266                     0        40  
379                     0        40  
497                     0        40  
614                     0        40  
727                     0        40  
843          

In [11]:
print("Number of records with a difference of -40:")
print((difference == -40).sum())

print("\nRecords by station:")
print(
    bikes.loc[difference == -40, "station_id"]
    .value_counts()
    .head(10)
)

Number of records with a difference of -40:
27715

Records by station:
station_id
51     26218
113      821
56       390
89       285
69         1
Name: count, dtype: int64


In [12]:
station_51 = bikes[bikes["station_id"] == 51]

print("Total records:", len(station_51))
print("Records with zero bikes and zero docks:",
      ((station_51["num_bikes_available"] == 0) &
       (station_51["num_docks_available"] == 0)).sum())
print("Station capacity values:")
print(station_51["capacity"].value_counts())

Total records: 26218
Records with zero bikes and zero docks: 26218
Station capacity values:
capacity
40    26218
Name: count, dtype: int64


In [13]:
print("First report:", station_51["last_reported"].min())
print("Last report:", station_51["last_reported"].max())

First report: 2025-05-01 00:10:00
Last report: 2025-10-31 23:55:00


In [14]:
print(bikes["last_reported"].dtype)

object


In [15]:
bikes["last_reported"] = pd.to_datetime(bikes["last_reported"])

print("Column type:", bikes["last_reported"].dtype)
print("Date range:", bikes["last_reported"].min(), "to", bikes["last_reported"].max())

Column type: datetime64[ns]
Date range: 2025-05-01 00:05:00 to 2025-10-31 23:55:00


In [16]:
# Remove columns that contain only missing values.
bikes = bikes.dropna(axis=1, how="all")

print("Dataset shape:", bikes.shape)
print("Remaining missing values:", bikes.isnull().sum().sum())
print("Columns:", list(bikes.columns))

Dataset shape: (3752455, 14)
Remaining missing values: 0
Columns: ['system_id', 'last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'address', 'lat', 'lon', 'capacity', 'availability_total']


In [17]:
print("Bike availability:")
print(bikes["num_bikes_available"].describe())

print("\nDock availability:")
print(bikes["num_docks_available"].describe())

print("\nStation capacity:")
print(bikes["capacity"].describe())

print("\nStation status values:")
print(bikes[["is_installed", "is_renting", "is_returning"]].nunique())

Bike availability:
count    3.752455e+06
mean     1.186609e+01
std      9.907475e+00
min      0.000000e+00
25%      3.000000e+00
50%      1.000000e+01
75%      1.800000e+01
max      4.000000e+01
Name: num_bikes_available, dtype: float64

Dock availability:
count    3.752455e+06
mean     1.961064e+01
std      1.138085e+01
min      0.000000e+00
25%      1.100000e+01
50%      1.900000e+01
75%      2.900000e+01
max      4.100000e+01
Name: num_docks_available, dtype: float64

Station capacity:
count    3.752455e+06
mean     3.185421e+01
std      7.479784e+00
min      1.600000e+01
25%      2.900000e+01
50%      3.000000e+01
75%      4.000000e+01
max      4.000000e+01
Name: capacity, dtype: float64

Station status values:
is_installed    1
is_renting      1
is_returning    1
dtype: int64


The results show that there were around 12 bikes and 20 available docks on average in each record the number of bikes and docks changed across the records .. Station capacity ranged from 16 to 40 spaces, with an average of around 32, this shows that not all stations have the same capacity
The three status columns (is_installed, is_renting and is_returning) have only one value each since the values do not change, these columns will not be useful for the clustering analysis

In [18]:
# Check whether each station has more than one record at the same timestamp.
duplicate_keys = bikes.duplicated(
    subset=["station_id", "last_reported"]
)

print("Duplicate station-timestamp records:", duplicate_keys.sum())

Duplicate station-timestamp records: 0


I checked whether the same station had more than one record at the same time the result was zero duplicate records, so I did not need to remove any rows for this reason.

In [19]:
print("Number of unique values per column:")
print(bikes.nunique())

Number of unique values per column:
system_id                  1
last_reported          52141
station_id               115
num_bikes_available       41
num_docks_available       42
is_installed               1
is_renting                 1
is_returning               1
name                     115
address                  115
lat                      115
lon                      115
capacity                  17
availability_total        43
dtype: int64


In [20]:
# Remove columns that do not vary across the dataset.
bikes = bikes.drop(
    columns=["system_id", "is_installed", "is_renting", "is_returning"]
)

print("Dataset shape:", bikes.shape)
print("Number of stations:", bikes["station_id"].nunique())
print("Number of columns:", len(bikes.columns))

Dataset shape: (3752455, 10)
Number of stations: 115
Number of columns: 10


### Removing Constant Columns

The columns `system_id`, `is_installed`, `is_renting` and `is_returning` contain only one unique value across the dataset, They do not help distinguish records or identify differences between stations, so they were remove from the analysis The station identifiers, timestamps, availability values and capacity were retained for further investigation

In [21]:
zero_zero = (
    (bikes["num_bikes_available"] == 0) &
    (bikes["num_docks_available"] == 0)
)

print("Total of 0/0 records:", zero_zero.sum())
print("\nRecords by station:")
print(
    bikes.loc[zero_zero, "station_id"]
    .value_counts()
    .head(10)
)

Total of 0/0 records: 30043

Records by station:
station_id
51     26218
25       831
113      821
26       810
39       410
56       390
89       285
13       267
70         9
69         1
Name: count, dtype: int64


### Investigating Zero Availability Records

Some records show zero available bikes and zero available docks at the same time, before treating these values as missing, I will compare their frequency across stations to identify whether the pattern is occasional or persistent

In [22]:
station_check = bikes.assign(zero_zero=zero_zero).groupby("station_id").agg(
    total_records=("last_reported", "size"),
    zero_zero_records=("zero_zero", "sum")
)

station_check["zero_zero_percentage"] = (
    station_check["zero_zero_records"] /
    station_check["total_records"] * 100
)

print(
    station_check
    .sort_values("zero_zero_records", ascending=False)
    .head(10)
    .round(2)
)

            total_records  zero_zero_records  zero_zero_percentage
station_id                                                        
51                  26218              26218                100.00
25                  31175                831                  2.67
113                 29228                821                  2.81
26                  32035                810                  2.53
39                  35155                410                  1.17
56                  33853                390                  1.15
89                  29033                285                  0.98
13                  30119                267                  0.89
70                  36539                  9                  0.02
69                  39962                  1                  0.00


### Handling Zero Availability Records

Station 51 was removed because it always reported zero available bikes and docks, even though its capacity was 40, I could not confirm from the dataset whether the station was closed or offline .. For the other stations, records with zero bikes and zero docks were treated as missing values This avoids assuming that these records represent normal station activity



In [23]:
# Exclude station 51 from the main analysis.
bikes = bikes[bikes["station_id"] != 51].copy()

# Treat simultaneous zero bike and dock counts as missing availability.
zero_zero = (
    (bikes["num_bikes_available"] == 0) &
    (bikes["num_docks_available"] == 0)
)

bikes.loc[
    zero_zero,
    ["num_bikes_available", "num_docks_available", "availability_total"]
] = np.nan

print("Dataset shape:", bikes.shape)
print("Number of stations:", bikes["station_id"].nunique())
print("Remaining 0/0 records:",
      ((bikes["num_bikes_available"] == 0) &
       (bikes["num_docks_available"] == 0)).sum())
print("Missing availability records:",
      bikes["num_bikes_available"].isna().sum())

Dataset shape: (3726237, 10)
Number of stations: 114
Remaining 0/0 records: 0
Missing availability records: 3825


In [24]:
# Check missing values after handling zero availability records.
print("Missing values by column:")
print(bikes.isnull().sum())

Missing values by column:
last_reported             0
station_id                0
num_bikes_available    3825
num_docks_available    3825
name                      0
address                   0
lat                       0
lon                       0
capacity                  0
availability_total     3825
dtype: int64



After handling the zero availability records, there are 3,825 missing values in the bike, dock and total availability columns, the other columns have no missing values.
These missing values come from records where both bikes and docks were reported as zero, I will keep them for now and consider how to handle them before the analysis

In [25]:
# Check the range of the numerical columns.
print("Bike availability:")
print(bikes["num_bikes_available"].min(), "-", bikes["num_bikes_available"].max())

print("\nDock availability:")
print(bikes["num_docks_available"].min(), "-", bikes["num_docks_available"].max())

print("\nStation capacity:")
print(bikes["capacity"].min(), "-", bikes["capacity"].max())

print("\nTotal availability:")
print(bikes["availability_total"].min(), "-", bikes["availability_total"].max())

Bike availability:
0.0 - 40.0

Dock availability:
0.0 - 41.0

Station capacity:
16 - 40

Total availability:
1.0 - 42.0


The bike and dock availability values do not contain negative numbers. station capacity ranges from 16 to 40

However, the maximum number of available docks is 41, and the maximum total availability is 42 .. These values are higher than the maximum station capacity, so I will check these records before deciding whether any changes are needed.

In [26]:
# Check records where available docks exceed station capacity.
docks_over_capacity = bikes[
    bikes["num_docks_available"] > bikes["capacity"]
]

print("Records where docks exceed capacity:", len(docks_over_capacity))

print("\nStation IDs:")
print(docks_over_capacity["station_id"].value_counts().head(10))

print("\nExamples:")
print(
    docks_over_capacity[
        ["last_reported", "station_id", "name",
         "num_docks_available", "capacity"]
    ].head(10)
)

Records where docks exceed capacity: 1

Station IDs:
station_id
49    1
Name: count, dtype: int64

Examples:
              last_reported  station_id          name  num_docks_available  \
2940397 2025-10-22 17:30:00          49  GUILD STREET                 41.0   

         capacity  
2940397        40  


Only one record showed more available docks than the station capacity. It was station 49 (GUILD STREET), with 41 available docks and a capacity of 40 on 22 October 2025 at 17:30.
Since this happened only once, I will keep the original value for now and note it as a possible data issue

In [28]:
# Check whether total availability matches station capacity.
availability_check = bikes[
    bikes["availability_total"].notna()
].copy()

availability_check["capacity_difference"] = (
    availability_check["availability_total"]
    - availability_check["capacity"]
)

print("Records checked:", len(availability_check))
print(
    "Records where total availability differs from capacity:",
    (availability_check["capacity_difference"] != 0).sum()
)

print("\nMost common differences:")
print(availability_check["capacity_difference"].value_counts().head(10))

Records checked: 3722412
Records where total availability differs from capacity: 166356

Most common differences:
capacity_difference
 0.0     3556056
-1.0      132487
-2.0       19685
-5.0        6712
-3.0        4954
-4.0        1203
-20.0        411
-6.0         301
-28.0         65
-39.0         64
Name: count, dtype: int64


Most records have a total availability that matches the station capacity. However, 166,356 records show a difference.
The most common difference is -1, meaning that the number of available bikes and docks is one less than the station capacity some records have larger differences, but I cannot confirm the reason from this dataset alone, so I will keep the original values for now

In [30]:
# Check whether station capacity stays consistent over time.
capacity_check = bikes.groupby("station_id")["capacity"].nunique()

print("Stations with more than one capacity value:")
print(capacity_check[capacity_check > 1])

print("\nNumber of stations with consistent capacity:")
print((capacity_check == 1).sum())

Stations with more than one capacity value:
Series([], Name: capacity, dtype: int64)

Number of stations with consistent capacity:
114


In [31]:
# Check for decimal values in bike and dock availability.
print("Bike availability decimal values:",
      (bikes["num_bikes_available"].dropna() % 1 != 0).sum())

print("Dock availability decimal values:",
      (bikes["num_docks_available"].dropna() % 1 != 0).sum())

Bike availability decimal values: 0
Dock availability decimal values: 0


I checked whether the bike and dock availability columns contain decimal values. Both checks returned zero, so all recorded availability values are whole numbers